# Módulo 2 — Tendência e sazonalidade

## Seção 2 — Sazonalidade em séries temporais

**Estado:** primeira versão, para revisão matemática e pedagógica.  
**Integração:** teoria, exemplos e Python no mesmo documento.  
**Referências utilizadas:** `mapeamento-conteudos-series-temporais.md`, especialmente os registros sobre componentes de séries temporais e mínimos quadrados para sazonalidade. As deduções, os exemplos numéricos e o código a seguir são complementações didáticas elaboradas para esta seção, e **não** transcrições das videoaulas.

### 1. Motivação: o que é sazonalidade?

Imagine uma loja com vendas mensais que aumentam regularmente em dezembro e diminuem em fevereiro. Quando essas diferenças se repetem em intervalos conhecidos, dizemos que há um **padrão sazonal**. O mesmo tipo de estrutura pode aparecer em temperaturas mensais, consumo de energia por hora ou número de atendimentos por dia da semana.

O **período sazonal** $s$ corresponde ao número de observações por ciclo. Em dados mensais com padrão anual, $s=12$; em dados trimestrais com padrão anual, $s=4$; em dados diários com padrão semanal, $s=7$; em dados horários com padrão diário, $s=24$. Essas interpretações dependem de uma frequência de observação regular.

Sazonalidade não significa repetição exata dos valores observados. A estrutura sistemática pode repetir-se, enquanto perturbações aleatórias provocam diferenças entre ciclos.

### 2. Sazonalidade determinística

Uma função sazonal determinística $S_t$ de período $s$ satisfaz

$$
S_{t+s}=S_t,\quad \text{para todo }t.
$$

Por exemplo, se $s=4$ e os efeitos sazonais são $(-3,1,4,-2)$, então $S_1=-3$, $S_2=1$, $S_3=4$, $S_4=-2$, e $S_5=-3$, $S_6=1$, etc.

Um modelo aditivo simples é

$$
Z_t=\mu+S_t+\varepsilon_t,
$$

em que $\mu$ é um nível constante, $S_t$ é um efeito conhecido pela posição no ciclo e $\varepsilon_t$ é uma perturbação aleatória. Sob as hipóteses

$$
E[\varepsilon_t]=0,\qquad \mathrm{Var}(\varepsilon_t)=\sigma^2<\infty,
$$

obtemos, pela linearidade da esperança,

$$
\begin{aligned}
E[Z_t]&=E[\mu+S_t+\varepsilon_t]\\
&=\mu+S_t+E[\varepsilon_t]\\
&=\mu+S_t.
\end{aligned}
$$

Como $\mu+S_t$ é determinístico, a adição desse termo não altera a variância:

$$
\mathrm{Var}(Z_t)=\mathrm{Var}(\varepsilon_t)=\sigma^2.
$$

**Interpretação.** A média pode variar periodicamente, embora a variância seja constante. Se $S_t$ não for constante, o processo não é fracamente estacionário, pois sua média depende de $t$. Constância da variância, isoladamente, não garante estacionariedade.

### 3. Exemplo numérico resolvido

Considere

$$
Z_t=20+S_t+\varepsilon_t,
$$

com período $s=4$ e efeitos sazonais $(-3,1,4,-2)$. Se $E[\varepsilon_t]=0$, as médias em cada posição do ciclo são:

| Trimestre | $S_t$ | $E[Z_t]=20+S_t$ |
|---|---:|---:|
| 1º | -3 | 17 |
| 2º | 1 | 21 |
| 3º | 4 | 24 |
| 4º | -2 | 18 |

Observe que

$$
-3+1+4-2=0.
$$

A condição de soma zero é uma **restrição de identificação** comum em modelos aditivos: fixa a interpretação de $\mu$ como média dos níveis sazonais do ciclo, evitando que uma constante arbitrária seja transferida de $\mu$ para todos os $S_t$ sem alterar o modelo.

De fato,

$$
\frac{17+21+24+18}{4}=20.
$$

Se a variância dos erros for $9$, cada trimestre terá variância igual a $9$, embora as médias sazonais sejam diferentes.

### 4. Implementação integrada em Python

O exemplo abaixo reproduz a equação do modelo, gera uma trajetória sintética e compara as observações com sua **média teórica periódica**. Requer Python 3, NumPy e Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gerador pseudoaleatório: a semente permite reproduzir a simulação.
rng = np.random.default_rng(42)

# Cinco anos de observações trimestrais.
n = 20
t = np.arange(1, n + 1)

# Modelo: Z_t = mu + S_t + epsilon_t
mu = 20
efeitos = np.array([-3, 1, 4, -2])

# Repetimos os quatro efeitos sazonais cinco vezes.
S = np.tile(efeitos, 5)

# Desvio padrão 3 -> variância teórica 3**2 = 9.
epsilon = rng.normal(loc=0, scale=3, size=n)

# Trajetória sintética e função média do modelo.
Z = mu + S + epsilon
media_teorica = mu + S

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, Z, "o-", label="Série simulada")
ax.plot(t, media_teorica, "--", label="Média teórica")
ax.set(xlabel="Trimestre", ylabel="Valor da série",
       title="Sazonalidade determinística: período de quatro trimestres")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


**Entendendo as operações:**

- `np.arange(1, n + 1)` constrói os índices inteiros de 1 a 20; o extremo superior é excluído.
- `np.array([-3, 1, 4, -2])` cria um vetor NumPy contendo os efeitos de cada trimestre.
- `np.tile(efeitos, 5)` repete o vetor cinco vezes: isso implementa matematicamente $S_{t+4}=S_t$.
- `rng.normal(loc=0, scale=3, size=n)` gera 20 valores pseudoaleatórios de uma distribuição normal de média 0 e **desvio padrão** 3. `scale` não é a variância; a variância é $9$.
- `Z = mu + S + epsilon` soma os vetores elemento a elemento e implementa diretamente o modelo.
- `media_teorica` não é uma média calculada a partir de 20 valores observados: é a função $E[Z_t]=\mu+S_t$ conhecida porque definimos o modelo de simulação.

**Resultado esperado.** Os valores observados oscilam ao redor de uma média periódica, mas não reproduzem exatamente os mesmos números a cada quatro trimestres. O gráfico **ilustra** as hipóteses e não prova estacionariedade ou não estacionariedade.

### 5. Introdução à sazonalidade estocástica

A sazonalidade determinística usa efeitos fixos. Uma dependência sazonal estocástica pode ser representada, por exemplo, por

$$
Z_t=\phi Z_{t-4}+\varepsilon_t.
$$

A observação atual depende do valor ocorrido quatro trimestres antes e de uma inovação. Esse exemplo é apresentado apenas para motivar a distinção: suas propriedades dependem de condições sobre $\phi$, dos erros e da especificação do processo. O estudo formal de modelos autorregressivos sazonais ocorrerá posteriormente.

| Característica | Sazonalidade determinística | Dependência sazonal estocástica |
|---|---|---|
| Representação ilustrativa | $S_{t+4}=S_t$ | $Z_t=\phi Z_{t-4}+\varepsilon_t$ |
| Componente sistemático | Efeitos fixos por posição no ciclo | Dependência probabilística em defasagens sazonais |
| Variação entre ciclos | Pode ocorrer devido ao ruído observado | Resulta também da própria dinâmica temporal |

Não se deve concluir que qualquer sazonalidade variável precise seguir essa equação; ela é apenas um mecanismo possível.

### 6. Exercício de fixação

Considere

$$
Z_t=10+S_t+\varepsilon_t,
$$

com período sazonal $s=3$, efeitos $S_1=-2$, $S_2=0$, $S_3=2$ e ruído branco com média zero e variância $4$.

1. Calcule $E[Z_1]$, $E[Z_2]$ e $E[Z_3]$.
2. Determine $E[Z_4]$ usando a periodicidade.
3. O processo é fracamente estacionário? Justifique especificando qual condição da definição é satisfeita ou violada.
4. Modifique o código para usar período $s=3$ e compare a trajetória com a função média teórica.

### 7. Continuidade e fontes

A próxima etapa será desenvolver a representação de sazonalidade por **variáveis indicadoras (dummies)** e por **componentes harmônicas (seno e cosseno)**, incluindo a estimação por mínimos quadrados e a correspondência com a implementação em Python.

**Fonte de planejamento efetivamente utilizada:** `mapeamento-conteudos-series-temporais.md`, registros “Aula 2 — Parte I — Tendência, Sazonalidade e Decomposição” e “Método dos Mínimos Quadrados: Sazonalidade (1 a 7)”. As fórmulas específicas desta seção e o experimento sintético são complementações didáticas; não há aqui atribuição a páginas não conferidas do livro de Morettin e Toloi.